# Домашнє завдання до Теми 5. Feature engineering на Olist: CTE, recursive CTE, Materialized View

Mykyta Oliinyk

Notebook виконується зверху вниз на PostgreSQL 16 через `pgserver`. Код не залежить від середовища: шляхи й драйвер однаково підходять для запуску локально і в Google Colab. Дані: повний Olist (варіант A, KaggleHub), stub-схема не використовується.

Де робота свідомо відходить від шаблонів умови:

| Що | Шаблон | Тут | Чому |
|---|---|---|---|
| Клієнт у pipeline | `customer_id` | ознаки рахуються на рівні `customer_unique_id`, рядок у результаті один на `customer_id`, додано колонку `customer_unique_id` | в Olist `customer_id` видається на кожне замовлення, тому `frequency` завжди 1 (див. перевірку в кінці Завдання 1) |
| EXPLAIN pipeline | лише Stage 0-2 | той самий скорочений запит плюс повний pipeline | без цього не можна відповісти, яка ознака найдорожча |
| Індекс для top-N | `(monetary DESC)` | замінено на `(monetary DESC NULLS LAST)` | індекс із шаблону не підходить під `ORDER BY monetary DESC NULLS LAST` (див. Завдання 5) |
| `print(dataset_dir)` | друкує шлях | друкується список файлів | шлях до кешу містить домашній каталог |
| `%pip install` | без `psycopg` | додано `psycopg[binary]` | SQLAlchemy 2.1 для адреси `postgresql://` обирає драйвер psycopg 3 |

## Завдання 1. Setup і typed-схема Olist

In [31]:
%%capture
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub "psycopg[binary]"

In [32]:
import os
import statistics
import time

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


def run_sql(sql):
    # DDL/DML: one transaction, committed on success.
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


def read_sql(sql):
    with engine.connect() as conn:
        return pd.read_sql(sql, conn)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Варіант A з умови: KaggleHub. Для цього датасету облікові дані Kaggle не знадобилися. Варіант B залишено в коментарі. Шлях `dataset_dir` не друкується, бо містить домашній каталог.

In [33]:
# Variant A: KaggleHub
import kagglehub
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(sorted(f for f in os.listdir(dataset_dir) if f.endswith('.csv')))

# Variant B: CSV from a course archive unpacked to data/olist/
# dataset_dir = os.path.join('data', 'olist')

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [34]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


Typed-таблиці: саме вони використовуються в усіх наступних завданнях. `DROP ... CASCADE` також прибирає MV з попереднього запуску, а в `hw5_referrals` лише FK; саму таблицю видаляє `DROP TABLE` у Завданні 3.

In [35]:
run_sql("""
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")

Перенесення з raw у typed. У CSV колонки написані з помилкою (`product_name_lenght`, `product_description_lenght`), тому в `SELECT` стоять фактичні назви, а в typed-схемі нормалізовані. `ANALYZE` у кінці дає планувальнику статистику для Завдання 5.

In [36]:
run_sql("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

ANALYZE olist_customers;
ANALYZE olist_orders;
ANALYZE olist_products;
ANALYZE olist_sellers;
ANALYZE olist_order_items;
""")

Smoke-test кількості рядків: запит з умови і порівняння з raw-таблицями.

In [37]:
smoke = read_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")
raw_counts = read_sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_raw FROM olist_customers_raw
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw;
""")
smoke = smoke.merge(raw_counts, on='table_name')
smoke['lost_rows'] = smoke['n_raw'] - smoke['n_rows']
smoke

,table_name,n_rows,n_raw,lost_rows
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,sellers,3095,3095,0


Усі п’ять typed-таблиць мають стільки ж рядків, скільки raw: `lost_rows` дорівнює 0, отже всі рядки пройшли перетворення типів, PK, FK і CHECK.

### Перевірка grain: що таке клієнт в Olist

Шаблони умови рахують ознаки на рівні `customer_id`. Перед цим варто перевірити, скільки замовлень припадає на один `customer_id`.

In [38]:
read_sql("""
WITH per_customer_id AS (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY customer_id
),
per_unique_id AS (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    'customer_id' AS grain,
    COUNT(*) AS n_keys,
    MAX(n_orders) AS max_orders,
    COUNT(*) FILTER (WHERE n_orders > 1) AS keys_with_2plus_orders
FROM per_customer_id
UNION ALL
SELECT
    'customer_unique_id',
    COUNT(*),
    MAX(n_orders),
    COUNT(*) FILTER (WHERE n_orders > 1)
FROM per_unique_id;
""")

,grain,n_keys,max_orders,keys_with_2plus_orders
0,customer_id,99441,1,0
1,customer_unique_id,96096,17,2997


На один `customer_id` припадає рівно одне замовлення. За шаблоном `frequency` була б 1 у всіх, `cancellation_rate` тільки 0 або 1, а rolling AVG просто дорівнював би `order_total`. Реальну людину ідентифікує `customer_unique_id`: 96 096 людей, з них 2 997 мають кілька замовлень, максимум 17.

Тому в Stage 1 і Stage 6 додано `JOIN olist_customers`, а всі ознаки рахуються на рівні `customer_unique_id`. Фінальний результат лишається з тим самим grain, що в умові: один рядок на `customer_id`, тож UNIQUE INDEX і lookup із Завдань 4-5 працюють без змін. Наслідок: у людини з кількома `customer_id` ці рядки мають однакові обчислені ознаки, тому `SUM(monetary)` у MV не можна рахувати без `DISTINCT ON (customer_unique_id)`. Для людей з одним замовленням значення збігаються з шаблоном (93 099 рядків), відрізняються лише 6 342 рядки людей із кількома замовленнями (99 441 − 96 096 + 2 997).

## Завдання 2. CTE-chain pipeline з ≥5 stages

Pipeline має вісім named stages (0-7), кожен виконує одну трансформацію. Текст CTE винесено в змінні, щоб MV і повний EXPLAIN запускали той самий код, а не його копію.

Рішення щодо визначень:

- `as_of_date` береться з даних (`MAX(order_purchase_timestamp)::DATE + 1`), а не з `CURRENT_DATE`.
- `frequency` і `monetary`, як у шаблоні, рахують усі замовлення незалежно від статусу: `frequency` є знаменником `cancellation_rate`, а внесок скасованих замовлень у `monetary` виміряно нижче.
- `avg_order_value_30d_recent` дорівнює rolling AVG на момент останнього замовлення клієнта (`rn_recent = 1`), а не на `as_of_date`.

In [39]:
PIPELINE_CTES = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status, keyed by the real customer
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_unique_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_unique_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        c.customer_unique_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY c.customer_unique_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
"""

PIPELINE_SELECT = """
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_unique_id = c.customer_unique_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_unique_id = c.customer_unique_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_unique_id = c.customer_unique_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_unique_id = c.customer_unique_id
"""

PIPELINE_SQL = PIPELINE_CTES + PIPELINE_SELECT

read_sql(PIPELINE_SQL + """
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
""")

,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,349509b216bd5ec11c5fae929fd13595,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
2,df55c14d1476a9a3467f131269c2477f,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
3,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
4,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
5,f48d464a0baaea338cb25f816991ab1f,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
6,3fd6777bbce08a352fddd04e4a7cc8f6,ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
7,05455dfa7cd02f13d132aa7a6a9729c6,4007669dec559734d6f53e029e360987,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
8,e0a2412720e9ea4f26c1ac985f6a7358,5d0a2980b292d049061542014e8960bf,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
9,24bbf5fd2f2e1b359ee7de94defc4a15,eebb5dda148d3893cdaf5b5ca3040ccb,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34


### Перевірка stages

Кожен stage можна запустити окремо, бо CTE-ланцюжок закінчується простим `SELECT` з потрібного етапу. Перевіряю кількість рядків і унікальність ключа.

In [40]:
stage_checks = [
    ('stage1_orders_total', 'order_id', 'один рядок на замовлення'),
    ('stage2_rfm', 'customer_unique_id', 'один рядок на людину'),
    ('stage3_cohort', 'customer_unique_id', 'один рядок на людину'),
    ('stage4_rolling_aov', 'order_id', 'один рядок на замовлення'),
    ('stage5_cancel_rate', 'customer_unique_id', 'один рядок на людину'),
    ('stage6_category_revenue', "customer_unique_id || '/' || product_category_name",
     'один рядок на пару людина-категорія'),
    ('stage7_preferred_category', 'customer_unique_id', 'один рядок на людину з товарами'),
]
rows = []
for stage, key, expected_grain in stage_checks:
    check = read_sql(PIPELINE_CTES + f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT {key}) AS n_keys
    FROM {stage};
    """)
    rows.append({'stage': stage, 'expected_grain': expected_grain,
                 'n_rows': int(check.loc[0, 'n_rows']), 'n_keys': int(check.loc[0, 'n_keys'])})
pd.DataFrame(rows)

,stage,expected_grain,n_rows,n_keys
0,stage1_orders_total,один рядок на замовлення,99441,99441
1,stage2_rfm,один рядок на людину,96096,96096
2,stage3_cohort,один рядок на людину,96096,96096
3,stage4_rolling_aov,один рядок на замовлення,99441,99441
4,stage5_cancel_rate,один рядок на людину,96096,96096
5,stage6_category_revenue,один рядок на пару людина-категорія,97915,97915
6,stage7_preferred_category,один рядок на людину з товарами,95420,95420


In [41]:
read_sql(PIPELINE_CTES + f"""
, final_features AS (
{PIPELINE_SELECT}
)
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT customer_id) AS n_customer_ids,
    COUNT(DISTINCT customer_unique_id) AS n_people,
    MIN(recency_days) AS min_recency,
    MAX(recency_days) AS max_recency,
    MAX(frequency) AS max_frequency,
    MIN(cancellation_rate) AS min_cancel_rate,
    MAX(cancellation_rate) AS max_cancel_rate,
    COUNT(*) FILTER (WHERE first_order_date IS NULL) AS null_first_order,
    COUNT(*) FILTER (WHERE preferred_category = '(none)') AS no_items,
    COUNT(*) FILTER (WHERE preferred_category = '(unknown)') AS unknown_category
FROM final_features;
""")

,n_rows,n_customer_ids,n_people,min_recency,max_recency,max_frequency,min_cancel_rate,max_cancel_rate,null_first_order,no_items,unknown_category
0,99441,99441,96096,1,774,17,0.0,1.0,0,685,1401


Фінальний результат: 99 441 рядок, 99 441 різний `customer_id`, тобто LEFT JOIN-и не розмножили рядки. `first_order_date` без NULL: кожен клієнт має замовлення, тому заглушка `recency_days = 9999` не спрацьовує. `(none)` означає, що в замовленнях клієнта немає жодної позиції, `(unknown)` позначає товари без категорії в Olist.

Перевірка на fan-out: сума `monetary` по одному рядку на людину має дорівнювати сумі `price + freight_value` у вихідній таблиці. Тут же видно внесок скасованих замовлень.

In [42]:
display(read_sql(PIPELINE_CTES + """
SELECT
    (SELECT SUM(monetary) FROM stage2_rfm) AS sum_monetary_per_person,
    (SELECT SUM(price + freight_value) FROM olist_order_items) AS sum_source_items;
"""))

display(read_sql(PIPELINE_CTES + """
SELECT
    order_status,
    COUNT(*) AS n_orders,
    COUNT(*) FILTER (WHERE order_total = 0) AS n_without_items,
    SUM(order_total) AS revenue,
    ROUND(100 * SUM(order_total) / SUM(SUM(order_total)) OVER (), 2) AS revenue_pct
FROM stage1_orders_total
GROUP BY order_status
ORDER BY revenue DESC;
"""))

,sum_monetary_per_person,sum_source_items
0,15843553.24,15843553.24


,order_status,n_orders,n_without_items,revenue,revenue_pct
0,delivered,96478,0,15419773.75,97.33
1,shipped,1107,1,177129.34,1.12
2,canceled,625,164,105885.72,0.67
3,processing,301,0,69394.11,0.44
4,invoiced,314,2,68988.75,0.44
5,unavailable,609,603,2140.49,0.01
6,approved,2,0,241.08,0.00
7,created,5,5,0.00,0.00


Суми збігаються, отже fan-out немає. Статуси `canceled` і `unavailable` дають менше 1% виручки (колонка `revenue_pct`), тому визначення з шаблону майже не зміщує `monetary`, але для задач, де важлива саме оплачена виручка, їх слід відфільтрувати. 775 замовлень без позицій мають `order_total = 0` і входять у `frequency` та rolling AVG.

Ручна перевірка rolling window на людині з найбільшою кількістю замовлень у межах 30 днів: кожне значення можна перерахувати за колонкою `order_total`.

In [43]:
read_sql(PIPELINE_CTES + """
, busiest AS (
    SELECT customer_unique_id
    FROM stage1_orders_total AS a
    WHERE EXISTS (
        SELECT 1
        FROM stage1_orders_total AS b
        WHERE b.customer_unique_id = a.customer_unique_id
          AND b.order_id <> a.order_id
          AND b.order_ts BETWEEN a.order_ts - INTERVAL '30 days' AND a.order_ts
    )
    GROUP BY customer_unique_id
    ORDER BY COUNT(*) DESC, customer_unique_id
    LIMIT 1
)
SELECT
    LEFT(s.customer_unique_id, 8) AS person,
    s.order_ts,
    s.order_total,
    s.avg_order_value_30d,
    s.rn_recent
FROM stage4_rolling_aov AS s
JOIN busiest AS b
    ON b.customer_unique_id = s.customer_unique_id
ORDER BY s.order_ts;
""")

,person,order_ts,order_total,avg_order_value_30d,rn_recent
0,8d50f5ea,2017-05-15 23:30:03,22.77,22.77,17
1,8d50f5ea,2017-06-18 22:56:48,51.75,51.75,16
2,8d50f5ea,2017-07-17 22:11:13,0.00,25.88,15
3,8d50f5ea,2017-07-18 23:10:58,21.77,10.89,14
4,8d50f5ea,2017-07-24 22:11:50,166.71,62.83,13
5,8d50f5ea,2017-08-05 08:59:43,81.89,67.59,12
6,8d50f5ea,2017-09-05 22:14:52,51.75,51.75,11
7,8d50f5ea,2017-10-18 23:25:04,37.77,37.77,10
8,8d50f5ea,2017-10-29 16:58:02,21.77,29.77,9
9,8d50f5ea,2017-11-22 20:01:53,21.77,21.77,8


## Завдання 3. Recursive CTE для referral chain

Olist не має referral-даних, тому `hw5_referrals` синтетична: перші 10 клієнтів за `customer_id` утворюють ланцюжок із 9 ребер. Бізнес-сенсу зв’язки не мають, таблиця потрібна для демонстрації обходу.

In [44]:
run_sql("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

read_sql("""
SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-11-06 19:01:07.885593+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-11-06 19:01:07.885593+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-11-06 19:01:07.885593+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-11-06 19:01:07.885593+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-11-06 19:01:07.885593+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-11-06 19:01:07.885593+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-11-06 19:01:07.885593+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-11-06 19:01:07.885593+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-11-06 19:01:07.885593+00:00


In [45]:
REFERRAL_CTE = """
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers not referred by anyone
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )
    UNION ALL
    -- RECURSIVE: step from a referrer to the customers they referred
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
"""

with pd.option_context('display.max_colwidth', None):
    display(read_sql(REFERRAL_CTE + """
    SELECT
        depth,
        COUNT(*) AS n_customers,
        MAX(path_text) AS sample_path
    FROM referral_depth
    GROUP BY depth
    ORDER BY depth;
    """))

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


In [46]:
read_sql(REFERRAL_CTE + """
SELECT
    (SELECT COUNT(*) FROM referral_depth WHERE depth = 1) AS n_roots,
    (SELECT COUNT(*) FROM referral_depth AS rd
      WHERE rd.depth = 1
        AND EXISTS (SELECT 1 FROM hw5_referrals AS r
                    WHERE r.referrer_customer_id = rd.customer_id)) AS roots_with_referrals,
    (SELECT MAX(depth) FROM referral_depth) AS max_depth,
    (SELECT COUNT(*) FROM referral_depth) - (SELECT COUNT(*) FROM olist_customers) AS extra_rows,
    (SELECT COUNT(*) FROM hw5_referrals AS r
      WHERE NOT EXISTS (SELECT 1 FROM referral_depth AS rd
                        WHERE rd.customer_id = r.referred_customer_id)) AS unreachable_referred;
""")

,n_roots,roots_with_referrals,max_depth,extra_rows,unreachable_referred
0,99432,1,10,0,0


Anchor за умовою бере всіх, кого ніхто не запросив, тому глибина 1 містить 99 432 корені, з яких лише один має рефералів; решта не беруть участі в програмі. Ланцюжок дає глибини 2-10 по одному клієнту, тобто `max_depth = 10` для 10 учасників. `extra_rows = 0` і `unreachable_referred = 0`: кожного клієнта відвідано рівно один раз.

### Що відбувається при циклі A → B → A

Цикл перевірено на окремому мінімальному графі у `VALUES`, щоб не псувати `hw5_referrals`. Порівнюються чотири варіанти: anchor з `NOT EXISTS`, anchor примусово з A із двома захистами, лише depth guard і стандартний `CYCLE`.

In [47]:
EDGES = "edges(referrer_id, referred_id) AS (VALUES ('A', 'B'), ('B', 'A'))"
NODES = "nodes(customer_id) AS (VALUES ('A'), ('B'))"

scenarios = {
    'NOT EXISTS anchor, path_ids + depth guard': f"""
        WITH RECURSIVE {EDGES}, {NODES},
        walk AS (
            SELECT n.customer_id, 1 AS depth, ARRAY[n.customer_id]::TEXT[] AS path_ids
            FROM nodes AS n
            WHERE NOT EXISTS (SELECT 1 FROM edges AS e WHERE e.referred_id = n.customer_id)
            UNION ALL
            SELECT e.referred_id, w.depth + 1, w.path_ids || e.referred_id
            FROM edges AS e
            JOIN walk AS w ON w.customer_id = e.referrer_id
            WHERE NOT e.referred_id = ANY(w.path_ids) AND w.depth < 50
        )
        SELECT * FROM walk""",
    'anchor = A, path_ids + depth guard': f"""
        WITH RECURSIVE {EDGES},
        walk AS (
            SELECT 'A'::TEXT AS customer_id, 1 AS depth, ARRAY['A']::TEXT[] AS path_ids
            UNION ALL
            SELECT e.referred_id, w.depth + 1, w.path_ids || e.referred_id
            FROM edges AS e
            JOIN walk AS w ON w.customer_id = e.referrer_id
            WHERE NOT e.referred_id = ANY(w.path_ids) AND w.depth < 50
        )
        SELECT * FROM walk""",
    'anchor = A, only depth guard': f"""
        WITH RECURSIVE {EDGES},
        walk AS (
            SELECT 'A'::TEXT AS customer_id, 1 AS depth, ARRAY['A']::TEXT[] AS path_ids
            UNION ALL
            SELECT e.referred_id, w.depth + 1, w.path_ids || e.referred_id
            FROM edges AS e
            JOIN walk AS w ON w.customer_id = e.referrer_id
            WHERE w.depth < 50
        )
        SELECT * FROM walk""",
    'anchor = A, CYCLE clause': f"""
        WITH RECURSIVE {EDGES},
        walk AS (
            SELECT 'A'::TEXT AS customer_id, 1 AS depth
            UNION ALL
            SELECT e.referred_id, w.depth + 1
            FROM edges AS e
            JOIN walk AS w ON w.customer_id = e.referrer_id
        ) CYCLE customer_id SET is_cycle USING cycle_path
        SELECT customer_id, depth FROM walk WHERE NOT is_cycle""",
}

rows = []
for name, query in scenarios.items():
    result = read_sql(query)
    rows.append({'scenario': name, 'n_rows': len(result),
                 'max_depth': None if result.empty else int(result['depth'].max())})
pd.DataFrame(rows)

,scenario,n_rows,max_depth
0,"NOT EXISTS anchor, path_ids + depth guard",0,NaN
1,"anchor = A, path_ids + depth guard",2,2.0
2,"anchor = A, only depth guard",50,50.0
3,"anchor = A, CYCLE clause",2,2.0


Висновки з таблиці:

- Якщо anchor стартує з клієнта циклу, без захистів запит не завершиться: кожен крок дає новий рядок із більшим `depth`, тож робочий набір ніколи не стає порожнім. Тому такий варіант не запускався.
- З `path_ids` обхід зупиняється на глибині 2: наступний крок повертає A, а A вже в масиві, робочий набір порожній, і рекурсія закінчується.
- Лише з depth guard запит завершується, але дає 50 рядків із хибними шляхами A → B → A → ... Тому `rd.depth < 50` лише страховка, а не логіка.
- `CYCLE` дає той самий результат, що й `path_ids`, але декларативно.
- Важливий нюанс: у чистому циклі немає кореня, тож anchor із `NOT EXISTS` не вибирає A і B взагалі (0 рядків). Через `PRIMARY KEY (referred_customer_id)` кожен клієнт має не більше одного запрошувача, тому цикл не може бути приєднаний до дерева, а завжди є окремою компонентою. Такі клієнти разом з усіма, кого вони запросили, мовчки випадають із результату, а не зациклюють запит. Це ловить перевірка `unreachable_referred` вище: зараз вона дорівнює 0. Отже, у запиті з `NOT EXISTS` масив `path_ids` є запасним захистом: він спрацює, якщо anchor стартує з конкретного клієнта або обмеження PK буде знято.

## Завдання 4. Materialized View з CONCURRENT REFRESH

MV загортає повний результат pipeline із Завдання 2: той самий текст `PIPELINE_SQL`, без скорочень. Після створення додаються UNIQUE INDEX на `customer_id` і `ANALYZE`.

In [48]:
run_sql(f"""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
{PIPELINE_SQL};

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

ANALYZE mv_olist_customer_features;
""")

display(read_sql("""
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
"""))

read_sql("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")

,n_rows
0,99441


,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,349509b216bd5ec11c5fae929fd13595,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
2,df55c14d1476a9a3467f131269c2477f,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
3,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
4,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31


Перевірка вимог `REFRESH ... CONCURRENTLY`: MV заповнена, індекс унікальний, побудований на колонці, а не на виразі, і не partial, тобто покриває всі рядки.

In [49]:
with pd.option_context('display.max_colwidth', None):
    display(read_sql("""
    SELECT
        i.relname AS index_name,
        m.ispopulated AS mv_populated,
        ix.indisunique AS is_unique,
        ix.indisvalid AS is_valid,
        ix.indpred IS NOT NULL AS is_partial,
        ix.indexprs IS NOT NULL AS is_expression,
        pg_get_indexdef(ix.indexrelid) AS definition
    FROM pg_index AS ix
    JOIN pg_class AS i ON i.oid = ix.indexrelid
    JOIN pg_matviews AS m ON m.matviewname = 'mv_olist_customer_features'
    WHERE ix.indrelid = 'mv_olist_customer_features'::regclass;
    """))

,index_name,mv_populated,is_unique,is_valid,is_partial,is_expression,definition
0,idx_mv_olist_customer_features_customer_id,True,True,True,False,False,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)


Concurrent refresh окремою клітинкою, після UNIQUE INDEX.

In [50]:
started = time.perf_counter()
run_sql("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;")
refresh_concurrent_ms = (time.perf_counter() - started) * 1000
print(f'REFRESH CONCURRENTLY: {refresh_concurrent_ms:.0f} ms')

REFRESH CONCURRENTLY: 7721 ms


Перевірки: MV збігається з pipeline в обох напрямках, а refresh реально підхоплює зміни. Друге перевіряється в транзакції з `ROLLBACK`: статус одного замовлення змінюється на `canceled`, до refresh MV показує старе значення, після refresh нове, а `ROLLBACK` повертає все назад.

In [51]:
display(read_sql(PIPELINE_CTES + f"""
, pipeline AS (
{PIPELINE_SELECT}
)
SELECT
    (SELECT COUNT(*) FROM (SELECT * FROM pipeline
                           EXCEPT SELECT * FROM mv_olist_customer_features) AS d) AS pipeline_minus_mv,
    (SELECT COUNT(*) FROM (SELECT * FROM mv_olist_customer_features
                           EXCEPT SELECT * FROM pipeline) AS d) AS mv_minus_pipeline;
"""))

CHECK_MV = """
SELECT customer_id, frequency, cancellation_rate
FROM mv_olist_customer_features
WHERE customer_id = (
    SELECT o.customer_id
    FROM olist_orders AS o
    WHERE o.order_status = 'delivered'
    ORDER BY o.order_id
    LIMIT 1
);
"""

conn = engine.connect()
target = conn.exec_driver_sql(CHECK_MV).one()
states = [('before update', target.frequency, target.cancellation_rate)]
order_id = conn.exec_driver_sql("""
    SELECT order_id FROM olist_orders WHERE customer_id = %s
""", (target.customer_id,)).scalar_one()
conn.exec_driver_sql("""
    UPDATE olist_orders SET order_status = 'canceled' WHERE order_id = %s
""", (order_id,))
MV_BY_ID = """
    SELECT frequency, cancellation_rate
    FROM mv_olist_customer_features
    WHERE customer_id = %s
"""
row = conn.exec_driver_sql(MV_BY_ID, (target.customer_id,)).one()
states.append(('after update, before refresh', row.frequency, row.cancellation_rate))
conn.exec_driver_sql("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features")
row = conn.exec_driver_sql(MV_BY_ID, (target.customer_id,)).one()
states.append(('after refresh', row.frequency, row.cancellation_rate))
conn.rollback()
row = conn.exec_driver_sql(MV_BY_ID, (target.customer_id,)).one()
states.append(('after rollback', row.frequency, row.cancellation_rate))
conn.close()

pd.DataFrame(states, columns=['state', 'frequency', 'cancellation_rate'])

,pipeline_minus_mv,mv_minus_pipeline
0,0,0


,state,frequency,cancellation_rate
0,before update,1,0.0000
1,"after update, before refresh",1,0.0000
2,after refresh,1,1.0000
3,after rollback,1,0.0000


MV повністю збігається з pipeline: обидві різниці `EXCEPT` дорівнюють 0. Без refresh MV не бачить зміну, після `REFRESH ... CONCURRENTLY` бачить, тобто дані в MV свіжі рівно настільки, наскільки свіжий останній refresh.

## Завдання 5. EXPLAIN ANALYZE + Reflection

Плани виводяться текстом у тому вигляді, який дає `EXPLAIN (ANALYZE, BUFFERS)`. Числа належать цьому запуску і змінюються між запусками.

In [52]:
def explain(sql):
    with engine.connect() as conn:
        rows = conn.exec_driver_sql('EXPLAIN (ANALYZE, BUFFERS)\n' + sql).fetchall()
    plan = '\n'.join(row[0] for row in rows)
    print(plan)
    return float(plan.rsplit('Execution Time: ', 1)[1].split(' ms')[0])


timings_ms = {}

### 1. EXPLAIN на CTE-pipeline

Спочатку запит із шаблону умови (Stage 0-2, top-10), адаптований до `customer_unique_id`.

In [53]:
timings_ms['Stage 0-2, top-10 (шаблон умови)'] = explain("""
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_unique_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

Limit  (cost=37938.93..37938.95 rows=10 width=65) (actual time=1140.686..1145.597 rows=10 loops=1)
  Buffers: shared hit=7418, temp read=2429 written=3166
  ->  Sort  (cost=37938.93..37939.43 rows=200 width=65) (actual time=1140.683..1145.592 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=7418, temp read=2429 written=3166
        ->  Subquery Scan on stage2_rfm  (cost=37929.11..37934.61 rows=200 width=65) (actual time=986.848..1119.081 rows=96096 loops=1)
              Buffers: shared hit=7418, temp read=2429 written=3166
              ->  HashAggregate  (cost=37929.11..37932.61 rows=200 width=69) (actual time=986.846..1108.841 rows=96096 loops=1)
                    Group Key: c.customer_unique_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7376kB
                    Buffers: shared hit=7418, temp read=

Шаблон покриває лише Stage 0-2, тобто не містить window functions і `preferred_category`. Для повного переобчислення запускаю весь pipeline.

In [54]:
timings_ms['Повний pipeline (Stage 0-7)'] = explain(PIPELINE_SQL)

Hash Left Join  (cost=100284.39..127272.70 rows=99441 width=183) (actual time=2900.228..3605.988 rows=99441 loops=1)
  Hash Cond: (c.customer_unique_id = cancel.customer_unique_id)
  Buffers: shared hit=15351, temp read=11103 written=11257
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=18906.55..30652.01 rows=112650 width=103) (actual time=555.376..886.461 rows=99441 loops=1)
          Group Key: c_2.customer_unique_id, o_1.order_id
          Buffers: shared hit=5609, temp read=1545 written=1549
          ->  Gather Merge  (cost=18906.55..28017.99 rows=66265 width=115) (actual time=555.357..749.945 rows=99684 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=5609, temp read=1545 written=1549
                ->  Partial GroupAggregate  (cost=17906.54..19563.17 rows=66265 width=115) (actual time=544.614..682.181 rows=49842 loops=2)
                      Group Key: c_2.customer_unique_id, o_1.order_id
    

### 2. EXPLAIN на lookup із MV

In [55]:
timings_ms['MV: lookup за customer_id'] = explain("""
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
""")

Nested Loop  (cost=0.83..8.91 rows=1 width=125) (actual time=0.043..0.045 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.47 rows=1 width=33) (actual time=0.023..0.023 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..5328.03 rows=99441 width=33) (actual time=0.021..0.022 rows=1 loops=1)
              Heap Fetches: 0
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=125) (actual time=0.015..0.015 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=4 read=4
Planning Time: 0.936 ms
Execution Time: 0.083 ms


Для чесного порівняння той самий lookup без MV: pipeline з фільтром на одного клієнта.

In [56]:
timings_ms['Pipeline: lookup за customer_id'] = explain(PIPELINE_SQL + """
WHERE c.customer_id = (SELECT MIN(customer_id) FROM olist_customers);
""")

Merge Right Join  (cost=78908.33..113246.64 rows=1 width=183) (actual time=1964.966..2142.748 rows=1 loops=1)
  Merge Cond: (ranked.customer_unique_id = c.customer_unique_id)
  Buffers: shared hit=13909, temp read=6514 written=8025
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=18906.55..30652.01 rows=112650 width=103) (actual time=538.348..853.689 rows=99441 loops=1)
          Group Key: c_2.customer_unique_id, o_1.order_id
          Buffers: shared hit=5609, temp read=1545 written=1549
          ->  Gather Merge  (cost=18906.55..28017.99 rows=66265 width=115) (actual time=538.296..719.961 rows=99685 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=5609, temp read=1545 written=1549
                ->  Partial GroupAggregate  (cost=17906.54..19563.17 rows=66265 width=115) (actual time=527.314..663.691 rows=49842 loops=2)
                      Group Key: c_2.customer_unique_id, o_1.order_id
            

### 3. Top-N із MV

Індекс із шаблону `(monetary DESC)` створюється з порядком за замовчуванням `NULLS FIRST`, а запит просить `NULLS LAST`. Спочатку запускаю точно як в умові.

In [57]:
TOP_N_SQL = """
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
"""

run_sql("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
""")
timings_ms['MV: top-10, індекс із шаблону'] = explain(TOP_N_SQL)

Limit  (cost=4961.01..4962.16 rows=10 width=125) (actual time=28.575..30.192 rows=10 loops=1)
  Buffers: shared hit=2148
  ->  Gather Merge  (cost=4961.01..11687.94 rows=58495 width=125) (actual time=28.572..30.187 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=2148
        ->  Sort  (cost=3961.00..4107.24 rows=58495 width=125) (actual time=25.179..25.183 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 29kB
              Buffers: shared hit=2148
              Worker 0:  Sort Method: top-N heapsort  Memory: 29kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2696.95 rows=58495 width=125) (actual time=0.009..7.240 rows=49720 loops=2)
                    Buffers: shared hit=2112
Planning:
  Buffers: shared hit=18 read=1
Planning Time: 0.341 ms
Execution Time: 30.235 ms


Індекс не використано. Виправлення: порядок NULL в індексі має збігатися із запитом. Непотрібний індекс видаляю, бо він лише сповільнює refresh.

In [58]:
run_sql("""
DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary_nl
    ON mv_olist_customer_features (monetary DESC NULLS LAST);
ANALYZE mv_olist_customer_features;
""")
timings_ms['MV: top-10, індекс NULLS LAST'] = explain(TOP_N_SQL)

Limit  (cost=0.42..1.53 rows=10 width=125) (actual time=0.041..0.058 rows=10 loops=1)
  Buffers: shared hit=9 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary_nl on mv_olist_customer_features  (cost=0.42..11044.03 rows=99441 width=125) (actual time=0.039..0.055 rows=10 loops=1)
        Buffers: shared hit=9 read=3
Planning:
  Buffers: shared hit=66 read=1
Planning Time: 0.450 ms
Execution Time: 0.081 ms


### 4. Вартість кожної ознаки

У повному плані Stage 1 матеріалізується один раз і читається кількома ознаками, тому час окремої ознаки з нього прочитати важко. Кожен ланцюжок запускається окремо: `WITH ... SELECT * FROM stageN`. CTE, які не читає фінальний запит, не виконуються, отже рахується лише сама ознака і її залежності. Кожен запит виконується 5 разів, у таблиці медіана `Execution Time` (`chain_ms`). `own_ms` показує вартість ознаки понад етап, від якого вона залежить: саме її додає ознака, коли цей етап уже пораховано. `preferred_category` не читає Stage 1 і має власний JOIN, тому її `own_ms` дорівнює `chain_ms`. Час у Colab шумний: `own_ms` у RFM вийшов від'ємним, бо ланцюжок виявився швидшим за власний Stage 1, а час `cancellation_rate` не підтверджується його `HashAggregate` у повному плані. Тому таблиця показує лише порядок величин, а висновки в Reflection спираються на план.

In [59]:
# feature: (stage, parent stage whose result it reuses)
FEATURES = {
    'order_total (спільна база)': ('stage1_orders_total', None),
    'RFM': ('stage2_rfm', 'stage1_orders_total'),
    'cohort_quarter': ('stage3_cohort', 'stage2_rfm'),
    'avg_order_value_30d': ('stage4_rolling_aov', 'stage1_orders_total'),
    'cancellation_rate': ('stage5_cancel_rate', 'stage1_orders_total'),
    'preferred_category': ('stage7_preferred_category', None),
}


def execution_ms(sql):
    with engine.connect() as conn:
        plan = conn.exec_driver_sql('EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)\n' + sql).scalar_one()
    return plan[0]['Execution Time']


chain_ms = {}
for feature, (stage, _parent) in FEATURES.items():
    sql = PIPELINE_CTES + f'SELECT * FROM {stage};'
    chain_ms[stage] = statistics.median(execution_ms(sql) for _ in range(5))

rows = []
for feature, (stage, parent) in FEATURES.items():
    rows.append({
        'feature': feature,
        'stage': stage,
        'parent': parent or '-',
        'chain_ms': round(chain_ms[stage], 1),
        'own_ms': round(chain_ms[stage] - (chain_ms[parent] if parent else 0), 1),
    })
feature_costs = pd.DataFrame(rows).sort_values('own_ms', ascending=False, ignore_index=True)
feature_costs

,feature,stage,parent,chain_ms,own_ms
0,cancellation_rate,stage5_cancel_rate,stage1_orders_total,2689.2,1485.0
1,order_total (спільна база),stage1_orders_total,-,1204.2,1204.2
2,preferred_category,stage7_preferred_category,-,1108.7,1108.7
3,avg_order_value_30d,stage4_rolling_aov,stage1_orders_total,1467.5,263.3
4,cohort_quarter,stage3_cohort,stage2_rfm,1305.0,143.4
5,RFM,stage2_rfm,stage1_orders_total,1161.6,-42.6


### 5. Зведення: час і розмір

In [60]:
summary = pd.DataFrame(
    [{'scenario': name, 'execution_ms': ms} for name, ms in timings_ms.items()]
)
summary['times_slower_than_mv_lookup'] = (
    summary['execution_ms'] / timings_ms['MV: lookup за customer_id']
).round(0).astype(int)
display(summary)
print(f'REFRESH CONCURRENTLY (Завдання 4): {refresh_concurrent_ms:.0f} ms')

read_sql("""
SELECT
    pg_size_pretty(pg_relation_size('mv_olist_customer_features')) AS mv_heap,
    pg_size_pretty(pg_indexes_size('mv_olist_customer_features')) AS mv_indexes,
    pg_size_pretty(pg_total_relation_size('mv_olist_customer_features')) AS mv_total;
""")

,scenario,execution_ms,times_slower_than_mv_lookup
0,"Stage 0-2, top-10 (шаблон умови)",1148.469,13837
1,Повний pipeline (Stage 0-7),3622.542,43645
2,MV: lookup за customer_id,0.083,1
3,Pipeline: lookup за customer_id,2152.770,25937
4,"MV: top-10, індекс із шаблону",30.235,364
5,"MV: top-10, індекс NULLS LAST",0.081,1


REFRESH CONCURRENTLY (Завдання 4): 7721 ms


,mv_heap,mv_indexes,mv_total
0,17 MB,7968 kB,24 MB


### 6. Reflection

**Найдорожча ознака.** Повний pipeline: 3622.542 ms. За планом найдорожчий ланцюжок має `avg_order_value_30d`: `Subquery Scan on aov` закінчується на 1496.7 ms, із них 979 ms на `CTE Scan` спільного Stage 1. Решту ділять Sort і window function: над `CTE Scan` стоїть `Sort Method: external merge Disk: 9448kB`, далі дві `WindowAgg` з `Incremental Sort` між ними, бо rolling AVG і `rn_recent` потребують різного порядку. Найдовший вузол: верхня `WindowAgg` (1164 → 1484 ms). `preferred_category` не читає Stage 1 і коштує 989 ms: власні `Hash Left Join` і Sort на диску перед `Finalize GroupAggregate`. `HashAggregate` у RFM і `cancellation_rate` дешевший: 250 і 211 ms. Таблиця ланцюжків у Colab шумна, тому висновок спирається на план.

**Кешування.** Варто. Lookup із MV йде через `Index Scan using idx_mv_olist_customer_features_customer_id`: `shared hit=8`, 0.083 ms. Той самий lookup через pipeline триває 2152.770 ms, повний прогін: `shared hit=15351`, `temp read=11103`. `REFRESH CONCURRENTLY` зайняв 7721 ms. `recency_days` змінюється раз на добу, тому доцільний щодобовий refresh, а для замороженого Olist достатньо on-demand. Top-10 з `NULLS LAST` переходить із `Parallel Seq Scan` на `Index Scan` (30.235 → 0.081 ms).

**VIEW.** Звичайний VIEW кращий, коли потрібні дані на момент запиту, а запити рідкі: наприклад, ручна перевірка клієнта, де вчорашній `cancellation_rate` неприйнятний, а дві секунди очікування нормально.

**Termination.** Recursive CTE завершується, коли recursive part повертає порожній набір. Якщо anchor стартує з клієнта циклу A → B → A, без захистів запит не завершиться. Умова `NOT r.referred_customer_id = ANY(rd.path_ids)` зупиняє обхід на глибині 2, а лише `rd.depth < 50` дає 50 рядків із хибними шляхами. Anchor із `NOT EXISTS` не вибирає A і B (0 рядків): кореня немає, тому клієнти циклу мовчки випадають. Альтернатива для `path_ids`: стандартний `CYCLE`.

**Trade-off.** Найважливіша latency: lookup із MV швидший за pipeline у десятки тисяч разів. Storage (24 MB) і recompute (7721 ms) дешеві, а добова затримка freshness прийнятна для batch-ознак.